In [1]:
import pandas as pd
import polars as pl

In [2]:
def dms_to_decimal(dms):
    """Convert DMS packed as ±DDMMSS or ±DDDMMSS to decimal degrees."""
    sign = -1 if str(dms).startswith("-") else 1
    dms = abs(int(dms))

    degrees = dms // 10000
    minutes = (dms % 10000) // 100
    seconds = dms % 100

    val = round(sign * (degrees + minutes / 60 + seconds / 3600), 4)
    return val

In [3]:
# Add decimal lat, lon and plot_id

df_plm = pl.read_csv("../data/raw/ICP/595_mm_20260227091917/mm_plm.csv", separator=";")

df_plm = df_plm.with_columns(
    [
        pl.col("latitude").map_elements(dms_to_decimal, return_dtype=pl.Float64).alias("Lat"),
        pl.col("longitude").map_elements(dms_to_decimal, return_dtype=pl.Float64).alias("Lon"),
    ]
)

df_plm = df_plm.with_columns(
    (
        pl.col("code_country").cast(pl.Utf8).str.zfill(2)
        + "."
        + pl.col("code_plot").cast(pl.Utf8).str.zfill(4)
    ).alias("plot_id")
)

df_plm.head()

survey_year,code_country,partner_code,code_plot,instrument_seq_nr,code_location,latitude,longitude,code_altitude,code_variable,position_vertical,code_recording,scanning_intervall,storing_intervall,sw_id,date_monitoring_first,date_monitoring_last,days_measuring,instrument_desc,other_obs,q_flag,change_date,code_line,line_nr,Lat,Lon,plot_id
i64,i64,i64,i64,i64,str,str,str,i64,str,f64,i64,f64,f64,str,str,str,i64,str,str,str,str,str,i64,f64,f64,str
1994,1,1,6,2,"""F""","""+501200""","""+034300""",4,"""AT""",1.5,50,60.0,30.0,null,"""1994-12-10""","""1994-12-31""",22,"""CHP 59""","""code_plot_instr: 6.02""",null,"""2009-12-06 22:40:00""","""MMFR1994-002378""",2378,50.2,3.7167,"""01.0006"""
1994,1,1,6,1,"""F""","""+501200""","""+034300""",4,"""PR""",1.0,50,60.0,30.0,null,"""1994-12-10""","""1994-12-31""",22,"""CHP 59""","""code_plot_instr: 6.01""",null,"""2009-12-06 22:40:00""","""MMFR1994-002189""",2189,50.2,3.7167,"""01.0006"""
1994,1,1,6,3,"""F""","""+501200""","""+034300""",4,"""RH""",1.5,50,60.0,30.0,null,"""1994-12-10""","""1994-12-31""",22,"""CHP 59""","""code_plot_instr: 6.03""",null,"""2009-12-06 22:40:00""","""MMFR1994-002188""",2188,50.2,3.7167,"""01.0006"""
1994,1,1,16,1,"""F""","""+481100""","""-013400""",2,"""PR""",1.0,50,60.0,30.0,null,"""1994-12-07""","""1994-12-31""",25,"""CHS 35""","""code_plot_instr: 16.01""",null,"""2009-12-06 22:40:00""","""MMFR1994-002379""",2379,48.1833,-1.5667,"""01.0016"""
1994,1,1,16,3,"""F""","""+481100""","""-013400""",2,"""RH""",1.5,50,60.0,30.0,null,"""1994-12-07""","""1994-12-31""",25,"""CHS 35""","""code_plot_instr: 16.03""",null,"""2009-12-06 22:40:00""","""MMFR1994-002380""",2380,48.1833,-1.5667,"""01.0016"""


In [4]:
# -------------------------------------------------------------------
# Data loading and preprocessing for ICP MM data
#
# This cell performs the following steps:
#
# 1. Reads the raw ICP MM CSV file (semicolon-separated).
# 2. Perform a sanity check to ensure that mean value is between min and max.
# 3. Generates a unique `plot_id` by zero-padding and combining
#    `code_country` and `code_plot` into the format CC.PPPP.
# 4. Converts the `date_observation` column from string to `pl.Date`.
# 5. Extracts `year` and `month` from the observation date to support
#    temporal filtering and aggregation.
# 6. Creates a `month_year` column in MM-YYYY format for monthly aggregation.
# 7. Filters out historical observations from 1960 and earlier, which are
#    outside the scope of the analysis.
#
# The resulting DataFrame is prepared for downstream grouping,
# aggregation, and pivot operations.
# -------------------------------------------------------------------

df = pl.read_csv("../data/raw/ICP/595_mm_20260227091917/mm_mem.csv", separator=";")

df = df.with_columns(
    pl.col("daily_min").cast(pl.Float64),
    pl.col("daily_mean").cast(pl.Float64),
    pl.col("daily_max").cast(pl.Float64),
)

df = df.with_columns(
    [
        pl.when(pl.col("daily_mean") < pl.col("daily_min"))
        .then(None)
        .otherwise(pl.col("daily_min"))
        .alias("daily_min"),
        pl.when(pl.col("daily_mean") > pl.col("daily_max"))
        .then(None)
        .otherwise(pl.col("daily_max"))
        .alias("daily_max"),
    ]
)

df = (
    df.with_columns(
        (
            pl.col("code_country").cast(pl.Utf8).str.zfill(2)
            + "."
            + pl.col("code_plot").cast(pl.Utf8).str.zfill(4)
        ).alias("plot_id")
    )
    .with_columns(
        pl.col("date_observation").str.strptime(pl.Date, "%Y-%m-%d").alias("date_observation")
    )
    .with_columns(
        [
            pl.col("date_observation").dt.year().alias("year"),
            pl.col("date_observation").dt.month().alias("month"),
        ]
    )
    .with_columns(pl.col("date_observation").dt.strftime("%m-%Y").alias("month_year"))
    .filter(pl.col("year") > 1960)
)

df.head()

survey_year,code_country,partner_code,code_plot,instrument_seq_nr,code_variable,date_observation,daily_mean,daily_min,daily_max,daily_completeness,code_data_origin,code_data_status,other_obs,q_flag,change_date,code_line,line_nr,plot_id,year,month,month_year
i64,i64,i64,i64,i64,str,date,f64,f64,f64,i64,i64,i64,str,str,str,str,i64,str,i32,i8,str
1994,1,1,6,3,"""RH""",1994-12-30,79.7,69.0,91.0,100,99,99,"""code_plot_instr: 6.03""",null,"""2009-12-06 22:36:46""","""MMFR1994-MAN-112559""",112559,"""01.0006""",1994,12,"""12-1994"""
1994,1,1,6,3,"""RH""",1994-12-31,90.1,84.0,94.0,100,99,99,"""code_plot_instr: 6.03""",null,"""2009-12-06 22:36:46""","""MMFR1994-MAN-112562""",112562,"""01.0006""",1994,12,"""12-1994"""
1994,1,1,6,3,"""RH""",1994-12-14,89.7,59.0,100.0,100,99,99,"""code_plot_instr: 6.03""",null,"""2009-12-06 22:36:46""","""MMFR1994-MAN-112610""",112610,"""01.0006""",1994,12,"""12-1994"""
1994,1,1,6,3,"""RH""",1994-12-15,92.8,73.0,98.0,100,99,99,"""code_plot_instr: 6.03""",null,"""2009-12-06 22:36:46""","""MMFR1994-MAN-112612""",112612,"""01.0006""",1994,12,"""12-1994"""
1994,1,1,6,3,"""RH""",1994-12-16,94.8,84.0,100.0,100,99,99,"""code_plot_instr: 6.03""",null,"""2009-12-06 22:36:46""","""MMFR1994-MAN-112615""",112615,"""01.0006""",1994,12,"""12-1994"""


In [5]:
df_ch = df.filter(pl.col("code_country") == 50)

df_ch.select("code_plot").unique().to_series().to_list()

[12, 9, 15, 18, 3, 6, 1, 13, 16, 4, 7, 10, 19, 11, 14, 5, 2, 8]

In [6]:
# There are duplicate observations for the same plot, variable, and date
# caused by multiple records with different `code_line` and `line_nr` values.
# These duplicates represent the same measurement context and should be
# consolidated into a single record.
#
# To resolve this, we group by the unique identifiers that define a single
# observation (country, plot, variable, date, plot_id, and month_year),
# and compute the mean of all remaining numeric columns across duplicates.
# This effectively averages over multiple entries for the same observation.
#
# After aggregation, we drop metadata and quality-control columns that are
# no longer meaningful once the duplicates have been collapsed.

df = (
    df.group_by(
        ["code_country", "code_plot", "code_variable", "date_observation", "plot_id", "month_year"]
    )
    .agg(
        [
            pl.all()
            .exclude(
                [
                    "code_country",
                    "code_plot",
                    "code_variable",
                    "date_observation",
                    "plot_id",
                    "month_year",
                ]
            )
            .mean()
        ]
    )
    .drop(
        [
            "code_data_origin",
            "code_data_status",
            "other_obs",
            "q_flag",
            "change_date",
            "code_line",
            "line_nr",
        ]
    )
)

df.head()

code_country,code_plot,code_variable,date_observation,plot_id,month_year,survey_year,partner_code,instrument_seq_nr,daily_mean,daily_min,daily_max,daily_completeness,year,month
i64,i64,str,date,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64
4,301,"""SR""",2014-02-15,"""04.0301""","""02-2014""",2014.0,3004.0,4.0,null,null,null,54.0,2014.0,2.0
4,705,"""WS""",2020-01-12,"""04.0705""","""01-2020""",2020.0,3304.0,29.0,1.0,null,6.43,100.0,2020.0,1.0
15,23,"""WS""",2009-11-02,"""15.0023""","""11-2009""",2009.0,15.0,47.0,2.8,null,8.7,100.0,2009.0,11.0
1,93,"""RH""",2023-01-02,"""01.0093""","""01-2023""",2023.0,1.0,3.0,72.2,24.0,100.0,100.0,2023.0,1.0
4,901,"""PR""",1999-08-05,"""04.0901""","""08-1999""",1999.0,2904.0,6.0,3.0,null,null,100.0,1999.0,8.0


In [7]:
df_pivoted = df.pivot(
    values=["daily_min", "daily_max", "daily_mean", "daily_completeness"],
    index=[
        "code_country",
        "code_plot",
        "plot_id",
        "month_year",
        "date_observation",
    ],
    on="code_variable",
)
df_pivoted = df_pivoted.select(
    [
        "code_country",
        "code_plot",
        "plot_id",
        "date_observation",
        "month_year",
        "daily_mean_PR",
        "daily_completeness_PR",
        "daily_mean_AT",
        "daily_min_AT",
        "daily_max_AT",
        "daily_completeness_AT",
        "daily_mean_RH",
        "daily_min_RH",
        "daily_max_RH",
        "daily_completeness_RH",
        "daily_mean_WS",
        "daily_max_WS",
        "daily_completeness_WS",
        "daily_mean_WD",
        "daily_completeness_WD",
        "daily_mean_SR",
        "daily_completeness_SR",
    ]
)
df_pivoted.head()

code_country,code_plot,plot_id,date_observation,month_year,daily_mean_PR,daily_completeness_PR,daily_mean_AT,daily_min_AT,daily_max_AT,daily_completeness_AT,daily_mean_RH,daily_min_RH,daily_max_RH,daily_completeness_RH,daily_mean_WS,daily_max_WS,daily_completeness_WS,daily_mean_WD,daily_completeness_WD,daily_mean_SR,daily_completeness_SR
i64,i64,str,date,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
4,301,"""04.0301""",2014-02-15,"""02-2014""",0.39,100.0,null,null,null,54.0,null,null,null,54.0,null,null,54.0,null,54.0,null,54.0
4,705,"""04.0705""",2020-01-12,"""01-2020""",0.45,100.0,1.1,-1.5,2.6,100.0,98.0,89.9,100.0,100.0,1.0,6.43,100.0,null,0.0,13.1,100.0
15,23,"""15.0023""",2009-11-02,"""11-2009""",null,null,2.65,1.5,4.1,100.0,93.1,83.5,100.0,100.0,2.8,8.7,100.0,162.6,100.0,10.3,100.0
1,93,"""01.0093""",2023-01-02,"""01-2023""",1.2,100.0,8.1,4.1,12.9,100.0,72.2,24.0,100.0,100.0,2.24,12.0,100.0,300.0,100.0,27.0,100.0
4,901,"""04.0901""",1999-08-05,"""08-1999""",3.0,100.0,18.3,14.3,24.2,100.0,73.5,45.4,96.1,100.0,1.9,6.7,100.0,120.0,100.0,90.8,100.0


In [8]:
# -------------------------------------------------------------------
# Variable-wise data completeness analysis

# This function analyzes data availability for a given variable
# (e.g., air temperature "AT") across plots and months.

# Execution steps:
# 1. Filter the input DataFrame to keep only rows corresponding to the
#    specified `code_variable`.
# 2. Report the number of unique month–year combinations available
#    for the selected variable, providing a quick overview of its
#    temporal coverage.
# 3. Group the filtered data by country, plot, and month–year.
# 4. For each group, check whether all values of `daily_mean`,
#    `daily_min`, and `daily_max` are missing (null).
# 5. Identify months where *all three* daily statistics are completely
#    null, indicating a full absence of usable data for that variable
#    in that plot and month.
# 6. Sort the resulting records chronologically by month–year.

# The function returns a DataFrame listing plot–month combinations
# where the selected variable has no valid observations at all.
# -------------------------------------------------------------------


def variable_wise_analysis(df: pl.DataFrame, variable: str):
    """Variable wise analysis for missing data."""
    tdf = df.filter(pl.col("code_variable") == variable)
    print(
        "Number of unique (month-year) data in ",
        variable,
        "is :",
        tdf.select(pl.col("month_year")).n_unique(),
    )
    null_months = (
        tdf.group_by(["code_country", "code_plot", "month_year"])
        .agg(
            [
                (pl.col("daily_mean").is_null().all()).alias("daily_mean_all_null"),
                (pl.col("daily_min").is_null().all()).alias("daily_min_all_null"),
                (pl.col("daily_max").is_null().all()).alias("daily_max_all_null"),
            ]
        )
        .filter(
            pl.col("daily_mean_all_null")
            & pl.col("daily_min_all_null")
            & pl.col("daily_max_all_null")
        )
    ).sort(by="month_year")
    return null_months


null_months = variable_wise_analysis(df, "AT")

null_months.head()

Number of unique (month-year) data in  AT is : 384


code_country,code_plot,month_year,daily_mean_all_null,daily_min_all_null,daily_max_all_null
i64,i64,str,bool,bool,bool
4,308,"""01-1994""",true,true,true
4,308,"""01-1996""",true,true,true
2,16,"""01-1996""",true,true,true
4,308,"""01-1997""",true,true,true
2,16,"""01-1997""",true,true,true


In [9]:
# -------------------------------------------------------------------
# Monthly aggregation of daily climate variables

# This block computes monthly summary statistics for each plot and
# variable based on daily observations.

# Execution steps:
# 1. Replace NaN values in daily statistics with nulls to ensure
#    correct aggregation behavior in Polars.
# 2. Group data by plot, variable, and calendar month
#    (`plot_id`, `code_variable`, `month_year`, `year`, `month`).
# 3. Compute monthly aggregates:
#    - `avg_daily_mean`: mean of daily mean values within the month
#    - `min_daily_min`: minimum of daily minimum values (monthly extreme)
#    - `max_daily_max`: maximum of daily maximum values (monthly extreme)
#    - `frost_days`: count of days with minimum temperature below 0°C
# 4. Sort the resulting monthly summaries chronologically by year
#    and month.

# The resulting DataFrame provides a compact monthly representation
# of daily climate dynamics for each plot and variable.
# -------------------------------------------------------------------

monthly_summary = (
    df.with_columns(
        [
            pl.col("daily_mean").fill_nan(None),
            pl.col("daily_min").fill_nan(None),
            pl.col("daily_max").fill_nan(None),
        ]
    )
    .group_by(
        [
            "plot_id",
            "code_country",
            "code_plot",
            "code_variable",
            "month_year",
            "year",
            "month",
        ]
    )
    .agg(
        [
            pl.col("daily_mean").mean().alias("avg_daily_mean"),
            pl.col("daily_min").min().alias("min_daily_min"),
            pl.col("daily_max").max().alias("max_daily_max"),
            (pl.col("daily_min") < 0).sum().alias("frost_days"),
            pl.col("daily_completeness").mean().alias("avg_completeness"),
        ]
    )
    .sort(["year", "month"])
)

monthly_summary.head()

plot_id,code_country,code_plot,code_variable,month_year,year,month,avg_daily_mean,min_daily_min,max_daily_max,frost_days,avg_completeness
str,i64,i64,str,str,f64,f64,f64,f64,f64,u32,f64
"""07.0011""",7,11,"""PR""","""01-1991""",1991.0,1.0,0.0,null,null,0,0.0
"""07.0010""",7,10,"""PR""","""01-1991""",1991.0,1.0,0.0,null,null,0,0.0
"""07.0001""",7,1,"""PR""","""01-1991""",1991.0,1.0,0.0,null,null,0,0.0
"""07.0109""",7,109,"""PR""","""01-1991""",1991.0,1.0,0.0,null,null,0,0.0
"""07.0011""",7,11,"""PR""","""02-1991""",1991.0,2.0,2.5,null,null,0,25.0


In [10]:
monthly_summary.filter((pl.col("code_variable") == "SR") & (pl.col("plot_id") == "02.0008"))

plot_id,code_country,code_plot,code_variable,month_year,year,month,avg_daily_mean,min_daily_min,max_daily_max,frost_days,avg_completeness
str,i64,i64,str,str,f64,f64,f64,f64,f64,u32,f64
"""02.0008""",2,8,"""SR""","""01-1996""",1996.0,1.0,628.967742,null,null,0,100.0
"""02.0008""",2,8,"""SR""","""02-1996""",1996.0,2.0,792.862069,null,null,0,100.0
"""02.0008""",2,8,"""SR""","""03-1996""",1996.0,3.0,1891.354839,null,null,0,100.0
"""02.0008""",2,8,"""SR""","""04-1996""",1996.0,4.0,3347.233333,null,null,0,100.0
"""02.0008""",2,8,"""SR""","""05-1996""",1996.0,5.0,2835.935484,null,null,0,100.0
…,…,…,…,…,…,…,…,…,…,…,…
"""02.0008""",2,8,"""SR""","""08-2025""",2025.0,8.0,213.845161,0.0,841.0,0,95.16129
"""02.0008""",2,8,"""SR""","""09-2025""",2025.0,9.0,136.173333,0.0,740.7,0,94.866667
"""02.0008""",2,8,"""SR""","""10-2025""",2025.0,10.0,72.096774,0.0,614.5,0,95.032258


In [11]:
# -------------------------------------------------------------------
# Temporal coverage analysis for each plot and variable
#
# This block evaluates the time span and completeness of climate data
# for every plot-variable combination, identifying gaps in the
# monthly time series.
#
# Execution steps:
# 1. Convert `month_year` from string format (MM-YYYY) to a proper
#    date type to enable chronological calculations.
# 2. Group the monthly summary data by plot and variable
#    (`plot_id`, `code_variable`).
# 3. For each plot-variable group, compute temporal coverage metrics:
#    - `n_rows`: total number of months with actual data available
#    - `min_month_year`: earliest recorded month in the time series
#    - `max_month_year`: latest recorded month in the time series
#    - `n_months`: expected total months if data were continuous
#      from the first to last observation (span calculation)
# 4. Calculate the expected number of months by converting the
#    date range into a month count (inclusive of both endpoints).
# 5. Sort the results by plot ID for easier interpretation.
#
# The resulting DataFrame provides a quick diagnostic to assess:
# - Data continuity (n_rows vs n_months)
# - Temporal extent for each plot-variable combination
# - Which plots have full vs. fragmented time series
# -------------------------------------------------------------------

plot_temporal_coverage = (
    monthly_summary.with_columns(
        # Ensure month_year is a proper date
        pl.col("month_year").str.strptime(pl.Date, "%m-%Y").alias("month_year")
    )
    .group_by("plot_id", "code_variable")
    .agg(
        [
            pl.len().alias("n_rows"),  # number of rows
            pl.col("month_year").min().alias("min_month_year"),  # earliest month
            pl.col("month_year").max().alias("max_month_year"),  # latest month
            (
                (pl.col("month_year").max().dt.year() - pl.col("month_year").min().dt.year()) * 12
                + (pl.col("month_year").max().dt.month() - pl.col("month_year").min().dt.month())
                + 1
            ).alias("n_months"),  # months span
        ]
    )
    .sort("plot_id")
)

plot_temporal_coverage.head()

plot_id,code_variable,n_rows,min_month_year,max_month_year,n_months
str,str,u32,date,date,i32
"""01.0003""","""AT""",146,1995-06-01,2008-07-01,158
"""01.0003""","""RH""",146,1995-06-01,2008-07-01,158
"""01.0003""","""PR""",146,1995-06-01,2008-07-01,158
"""01.0006""","""PR""",348,1994-12-01,2024-12-01,361
"""01.0006""","""AT""",348,1994-12-01,2024-12-01,361


In [12]:
# -------------------------------------------------------------------
# Identification of missing months in plot-variable time series
#
# This block builds on the temporal coverage summary to explicitly
# list which months are missing from each plot's data record.
#
# Execution steps:
# 1. Filter the `plot_temporal_coverage` DataFrame to retain only
#    plots where actual rows (`n_rows`) do not equal expected months
#    (`n_months`), indicating incomplete data.
# 2. For each plot-variable combination, retrieve the full list of
#    existing month-year values from the monthly summary.
# 3. Generate a complete sequence of all months between the minimum
#    and maximum recorded months using `pd.date_range`.
# 4. Compare the existing months against the complete sequence to
#    identify missing months.
# 5. Collect the count of missing months and the list of missing
#    month-year values for each plot.
# 6. Join the missing months information back to the original
#    `plot_temporal_coverage` DataFrame.
#
# The resulting DataFrame adds `n_missing_months` and `missing_months`
# columns, enabling detailed gap analysis and data quality assessment.
# -------------------------------------------------------------------


def missing_months(min_date, max_date, existing_months):
    """Create all months between min and max months."""
    all_months = pd.date_range(start=min_date, end=max_date, freq="MS").strftime("%m-%Y").to_list()

    missing = [d for d in all_months if d not in existing_months]
    return missing


incomplete_plots = plot_temporal_coverage.filter(pl.col("n_rows") != pl.col("n_months"))

# Collect month lists for each plot
missing_months_list = []

for row in plot_temporal_coverage.iter_rows(named=True):
    plot_id = row["plot_id"]
    code_var = row["code_variable"]
    min_month = row["min_month_year"]
    max_month = row["max_month_year"]

    # Existing months for this plot
    existing_months = (
        monthly_summary.filter(
            (pl.col("plot_id") == plot_id) & (pl.col("code_variable") == code_var)
        )
        .select("month_year")
        .to_series()
        .to_list()
    )

    missing = missing_months(min_month, max_month, existing_months)

    missing_months_list.append(
        {
            "plot_id": plot_id,
            "code_variable": code_var,
            "n_missing_months": len(missing),
            "missing_months": missing,
        }
    )

# Convert to DataFrame
missing_months_df = pl.DataFrame(missing_months_list)

plot_temporal_coverage = plot_temporal_coverage.join(
    missing_months_df, on=["plot_id", "code_variable"]
)

plot_temporal_coverage.head()

plot_id,code_variable,n_rows,min_month_year,max_month_year,n_months,n_missing_months,missing_months
str,str,u32,date,date,i32,i64,list[str]
"""01.0003""","""AT""",146,1995-06-01,2008-07-01,158,12,"[""01-2001"", ""02-2001"", … ""12-2001""]"
"""01.0003""","""RH""",146,1995-06-01,2008-07-01,158,12,"[""01-2001"", ""02-2001"", … ""12-2001""]"
"""01.0003""","""PR""",146,1995-06-01,2008-07-01,158,12,"[""01-2001"", ""02-2001"", … ""12-2001""]"
"""01.0006""","""PR""",348,1994-12-01,2024-12-01,361,13,"[""01-2001"", ""02-2001"", … ""12-2021""]"
"""01.0006""","""AT""",348,1994-12-01,2024-12-01,361,13,"[""01-2001"", ""02-2001"", … ""12-2021""]"


In [13]:
plot_temporal_coverage = plot_temporal_coverage.join(
    missing_months_df, on=["plot_id", "code_variable"]
)

plot_temporal_coverage.head()

plot_id,code_variable,n_rows,min_month_year,max_month_year,n_months,n_missing_months,missing_months,n_missing_months_right,missing_months_right
str,str,u32,date,date,i32,i64,list[str],i64,list[str]
"""01.0003""","""AT""",146,1995-06-01,2008-07-01,158,12,"[""01-2001"", ""02-2001"", … ""12-2001""]",12,"[""01-2001"", ""02-2001"", … ""12-2001""]"
"""01.0003""","""RH""",146,1995-06-01,2008-07-01,158,12,"[""01-2001"", ""02-2001"", … ""12-2001""]",12,"[""01-2001"", ""02-2001"", … ""12-2001""]"
"""01.0003""","""PR""",146,1995-06-01,2008-07-01,158,12,"[""01-2001"", ""02-2001"", … ""12-2001""]",12,"[""01-2001"", ""02-2001"", … ""12-2001""]"
"""01.0006""","""PR""",348,1994-12-01,2024-12-01,361,13,"[""01-2001"", ""02-2001"", … ""12-2021""]",13,"[""01-2001"", ""02-2001"", … ""12-2021""]"
"""01.0006""","""AT""",348,1994-12-01,2024-12-01,361,13,"[""01-2001"", ""02-2001"", … ""12-2021""]",13,"[""01-2001"", ""02-2001"", … ""12-2021""]"


In [14]:
plot_temporal_coverage.filter(pl.col("n_missing_months") == 0)

plot_id,code_variable,n_rows,min_month_year,max_month_year,n_months,n_missing_months,missing_months,n_missing_months_right,missing_months_right
str,str,u32,date,date,i32,i64,list[str],i64,list[str]
"""02.0001""","""WS""",328,1998-09-01,2025-12-01,328,0,[],0,[]
"""02.0001""","""RH""",328,1998-09-01,2025-12-01,328,0,[],0,[]
"""02.0001""","""AT""",328,1998-09-01,2025-12-01,328,0,[],0,[]
"""02.0001""","""ST""",328,1998-09-01,2025-12-01,328,0,[],0,[]
"""02.0001""","""WD""",36,2001-01-01,2003-12-01,36,0,[],0,[]
…,…,…,…,…,…,…,…,…,…
"""67.0005""","""SR""",144,2013-01-01,2024-12-01,144,0,[],0,[]
"""67.0005""","""RH""",144,2013-01-01,2024-12-01,144,0,[],0,[]
"""67.0005""","""AT""",144,2013-01-01,2024-12-01,144,0,[],0,[]


In [15]:
# -------------------------------------------------------------------
# Extraction and aggregation of sample plot data for climate simulation
#
# This block selects a single plot with high-quality, continuous data
# (plot_id = "02.0008") and prepares a consolidated monthly weather
# dataset suitable for climate analysis and simulation modeling.
#
# Execution steps:
# 1. Filter the main dataset to retain only the target plot
#    ("02.0008"), which was previously identified as having 360
#    consecutive months of complete data (no missing months).
#
# 2. Split the filtered data into separate DataFrames by variable type:
#    - `temp_df`: Air temperature data (code_variable = "AT")
#    - `prcp_df`: Precipitation data (code_variable = "PR")
#    - `srad_df`: Solar radiation data (code_variable = "SR")
#
# 3. Aggregate each variable to monthly resolution:
#    a. Temperature aggregation (`mtemp_df`):
#       - `tmp_ave`: mean daily temperature for the month
#       - `tmp_min`: minimum daily minimum temperature (monthly extreme)
#       - `tmp_max`: maximum daily maximum temperature (monthly extreme)
#       - `frost_days`: count of days with minimum temperature below 0°C
#
#    b. Precipitation aggregation (`mprcp_df`):
#       - `prcp`: sum of daily precipitation for the month
#
#    c. Solar radiation aggregation (`msrad_df`):
#       - `srad`: mean daily solar radiation for the month
#
# 4. Merge the three aggregated DataFrames on `month_year` to create
#    a unified monthly weather dataset.
#
# 5. Perform post-aggregation processing:
#    a. Convert `month_year` from string (MM-YYYY) to proper date type
#    b. Extract `year` and `month` components for temporal sorting
#    c. Select and reorder columns to a clean, logical structure:
#       [year, month, tmp_ave, tmp_min, tmp_max, frost_days, prcp, srad]
#    d. Sort chronologically by year and month
#    e. Drop any rows with null values to ensure data completeness
#
# 6. Convert the final Polars DataFrame to pandas format for potential
#    export or compatibility with other libraries (optional Excel export
#    is commented out).
#
# The resulting `weather_df` provides a clean, complete monthly climate
# time series for the selected plot, ready for:
# - Climate trend analysis
# - Crop growth modeling
# - Statistical simulation
# - Machine learning applications
# -------------------------------------------------------------------


# plot_id = "02.0008" has 360 months of consecutive data,
# so we select this plot for sample simulations

plot_df = df.filter(pl.col("plot_id") == "02.0008")

temp_df = plot_df.filter(pl.col("code_variable") == "AT")

prcp_df = plot_df.filter(pl.col("code_variable") == "PR")

srad_df = plot_df.filter(pl.col("code_variable") == "SR")

mtemp_df = temp_df.group_by(["month_year"]).agg(
    pl.col("daily_mean").mean().alias("tmp_ave"),
    pl.col("daily_min").min().alias("tmp_min"),
    pl.col("daily_max").max().alias("tmp_max"),
    (pl.col("daily_min") < 0).sum().alias("frost_days"),
)

mprcp_df = prcp_df.group_by("month_year").agg(pl.col("daily_mean").sum().alias("prcp"))

msrad_df = srad_df.group_by("month_year").agg(pl.col("daily_mean").mean().alias("srad"))

weather_df = mtemp_df.join(mprcp_df, on="month_year").join(msrad_df, on="month_year")

weather_df = (
    weather_df.with_columns(
        [pl.col("month_year").str.strptime(pl.Date, "%m-%Y").alias("month_year")]
    )
    .with_columns(
        [
            pl.col("month_year").dt.year().alias("year"),
            pl.col("month_year").dt.month().alias("month"),
        ]
    )
    .select(["year", "month", "tmp_ave", "tmp_min", "tmp_max", "frost_days", "prcp", "srad"])
    .sort(by=["year", "month"])
    .drop_nulls()
    .to_pandas()
)

# weather_df.to_excel("../data/intermediate/weather_data.xlsx", index=False)

weather_df

,year,month,tmp_ave,tmp_min,tmp_max,frost_days,prcp,srad
0,1996,1,3.067742,-5.50,13.85,12,20.0,628.967742
1,1996,2,2.032759,-6.70,10.30,14,110.5,792.862069
2,1996,3,3.490323,-7.05,18.30,19,23.6,1891.354839
3,1996,4,9.005000,-4.65,27.50,10,6.9,3347.233333
4,1996,5,10.140323,-3.40,28.20,3,79.2,2835.935484
...,...,...,...,...,...,...,...,...
355,2025,8,17.617742,5.55,33.35,0,26.3,213.845161
356,2025,9,13.885000,4.35,27.45,0,73.8,136.173333
357,2025,10,10.619355,2.35,18.25,0,91.1,72.096774
358,2025,11,6.713333,-7.10,16.20,8,79.5,36.536667


In [16]:
plot_temporal_coverage.filter(pl.col("plot_id") == "50.0018")

plot_id,code_variable,n_rows,min_month_year,max_month_year,n_months,n_missing_months,missing_months,n_missing_months_right,missing_months_right
str,str,u32,date,date,i32,i64,list[str],i64,list[str]
"""50.0018""","""WS""",121,2013-01-01,2023-01-01,121,0,[],0,[]
"""50.0018""","""AT""",121,2013-01-01,2023-01-01,121,0,[],0,[]
"""50.0018""","""SR""",121,2013-01-01,2023-01-01,121,0,[],0,[]
"""50.0018""","""RH""",121,2013-01-01,2023-01-01,121,0,[],0,[]
"""50.0018""","""WD""",121,2013-01-01,2023-01-01,121,0,[],0,[]
"""50.0018""","""PR""",121,2013-01-01,2023-01-01,121,0,[],0,[]


In [17]:
# Plot: Davos

plot_df = df.filter(pl.col("plot_id") == "50.0018")

temp_df = plot_df.filter(pl.col("code_variable") == "AT")

prcp_df = plot_df.filter(pl.col("code_variable") == "PR")

srad_df = plot_df.filter(pl.col("code_variable") == "SR")

mtemp_df = temp_df.group_by(["month_year"]).agg(
    pl.col("daily_mean").mean().alias("tmp_ave"),
    pl.col("daily_min").min().alias("tmp_min"),
    pl.col("daily_max").max().alias("tmp_max"),
    (pl.col("daily_min") < 0).sum().alias("frost_days"),
)

mprcp_df = prcp_df.group_by("month_year").agg(pl.col("daily_mean").sum().alias("prcp"))

msrad_df = srad_df.group_by("month_year").agg(pl.col("daily_mean").mean().alias("srad"))

weather_df = mtemp_df.join(mprcp_df, on="month_year").join(msrad_df, on="month_year")

weather_df = (
    weather_df.with_columns(
        [pl.col("month_year").str.strptime(pl.Date, "%m-%Y").alias("month_year")]
    )
    .with_columns(
        [
            pl.col("month_year").dt.year().alias("year"),
            pl.col("month_year").dt.month().alias("month"),
        ]
    )
    .select(["year", "month", "tmp_ave", "tmp_min", "tmp_max", "frost_days", "prcp", "srad"])
    .sort(by=["year", "month"])
    .drop_nulls()
    .to_pandas()
)

weather_df.to_excel("../data/intermediate/Davos_weather_data.xlsx", index=False)

weather_df.head()

,year,month,tmp_ave,tmp_min,tmp_max,frost_days,prcp,srad
0,2013,1,-4.078065,-17.8,7.9,31,22.0,59.883871
1,2013,2,-7.617500,-18.4,5.5,28,30.3,105.521429
2,2013,3,-1.813871,-16.6,9.5,30,19.3,143.580645
3,2013,4,3.561000,-12.0,15.6,16,24.4,201.916667
4,2013,5,4.560323,-3.8,15.5,11,89.7,179.774194


In [18]:
plot_df = df.filter(pl.col("plot_id") == "51.0001")

temp_df = plot_df.filter(pl.col("code_variable") == "AT")

prcp_df = plot_df.filter(pl.col("code_variable") == "PR")

srad_df = plot_df.filter(pl.col("code_variable") == "SR")

mtemp_df = temp_df.group_by(["month_year"]).agg(
    pl.col("daily_mean").mean().alias("tmp_ave"),
    pl.col("daily_min").min().alias("tmp_min"),
    pl.col("daily_max").max().alias("tmp_max"),
    (pl.col("daily_min") < 0).sum().alias("frost_days"),
)

mprcp_df = prcp_df.group_by("month_year").agg(pl.col("daily_mean").sum().alias("prcp"))

msrad_df = srad_df.group_by("month_year").agg(pl.col("daily_mean").mean().alias("srad"))

weather_df = mtemp_df.join(mprcp_df, on="month_year").join(msrad_df, on="month_year")

weather_df = (
    weather_df.with_columns(
        [pl.col("month_year").str.strptime(pl.Date, "%m-%Y").alias("month_year")]
    )
    .with_columns(
        [
            pl.col("month_year").dt.year().alias("year"),
            pl.col("month_year").dt.month().alias("month"),
        ]
    )
    .select(["year", "month", "tmp_ave", "tmp_min", "tmp_max", "frost_days", "prcp", "srad"])
    .sort(by=["year", "month"])
    .drop_nulls()
    .to_pandas()
)

# weather_df.to_excel("../data/intermediate/weather_data.xlsx", index=False)

weather_df.head()

,year,month,tmp_ave,tmp_min,tmp_max,frost_days,prcp,srad
0,2004,1,-4.642581,-14.65,4.25,30,40.6,29.619677
1,2004,2,-0.639483,-14.70,12.65,23,51.4,63.400000
2,2004,3,2.585161,-12.85,18.55,16,40.0,102.556129
3,2004,4,8.927167,1.25,20.20,0,55.6,188.674000
4,2004,5,11.079839,1.90,21.15,0,87.8,251.282258


In [19]:
# -------------------------------------------------------------------
# Summary table for weather_df: completeness and gap analysis
#
# This block generates a comprehensive summary of the weather_df
# dataset, evaluating data completeness, identifying gaps, and
# assessing temporal continuity.
# -------------------------------------------------------------------

# Create a summary DataFrame for weather_df
summary_stats = {
    "Metric": [
        "Total records (months)",
        "Start date",
        "End date",
        "Expected months (based on date range)",
        "Actual months present",
        "Missing months",
        "Completeness (%)",
        "Has gaps",
        "Number of gaps",
        "Largest gap (months)",
        "Average gap size (months)",
        "Unique years",
        "Unique months",
    ],
    "Value": [],
}

# Calculate basic metrics
total_records = len(weather_df)
start_date = weather_df["year"].min(), weather_df["month"].min()
end_date = weather_df["year"].max(), weather_df["month"].max()

# Calculate expected months if continuous
start_year, start_month = start_date
end_year, end_month = end_date
expected_months = (end_year - start_year) * 12 + (end_month - start_month) + 1

# Check for gaps
weather_df["date"] = pd.to_datetime(weather_df[["year", "month"]].assign(day=1))
all_months = pd.date_range(start=weather_df["date"].min(), end=weather_df["date"].max(), freq="MS")
existing_months = set(weather_df["date"])
miss_months = sorted(set(all_months) - existing_months)

# Calculate gap statistics
gaps = []
if missing_months:
    # Find consecutive missing months (gaps)
    missing_series = pd.Series(miss_months)
    if len(missing_series) > 0:
        diff = missing_series.diff().dt.days
        gap_starts = missing_series[diff > 31].tolist()
        gap_lengths = []

        if len(gap_starts) > 0:
            for i, gap_start in enumerate(gap_starts):
                if i < len(gap_starts) - 1:
                    gap_end = gap_starts[i + 1]
                    gap_months = len(
                        missing_series[(missing_series >= gap_start) & (missing_series < gap_end)]
                    )
                else:
                    gap_months = len(missing_series[missing_series >= gap_start])
                gap_lengths.append(gap_months)
        else:
            gap_lengths = [len(missing_series)]

        gaps = gap_lengths
    else:
        gaps = [len(miss_months)]

# Count years with complete data
weather_df["year_month"] = (
    weather_df["year"].astype(str) + "-" + weather_df["month"].astype(str).str.zfill(2)
)
complete_years = weather_df.groupby("year")["month"].nunique()
years_complete = complete_years[complete_years == 12].tolist()

# Populate summary values
summary_stats["Value"].extend(
    [
        total_records,
        f"{start_date[0]}-{start_date[1]:02d}",
        f"{end_date[0]}-{end_date[1]:02d}",
        expected_months,
        total_records,
        len(miss_months),
        f"{(total_records / expected_months * 100):.1f}%",
        "Yes" if missing_months else "No",
        len(gaps),
        max(gaps) if gaps else 0,
        f"{sum(gaps) / len(gaps):.1f}" if gaps else "N/A",
        weather_df["year"].nunique(),
        weather_df["month"].nunique(),
    ]
)

summary_df = pd.DataFrame(summary_stats)
print("WEATHER DATASET COMPLETENESS SUMMARY")
summary_df

WEATHER DATASET COMPLETENESS SUMMARY


,Metric,Value
0,Total records (months),212
1,Start date,2004-01
2,End date,2021-12
3,Expected months (based on date range),216
4,Actual months present,212
5,Missing months,4
6,Completeness (%),98.1%
7,Has gaps,Yes
8,Number of gaps,1
9,Largest gap (months),4


In [21]:
# Detailed missing months analysis

print("MISSING MONTHS ANALYSIS")

if missing_months:
    # Group missing months by year
    missing_df = pd.DataFrame({"missing_date": miss_months})
    missing_df["year"] = missing_df["missing_date"].dt.year
    missing_df["month"] = missing_df["missing_date"].dt.month
    missing_by_year = missing_df.groupby("year").size().sort_index()

    print("\nMissing months by year:")
    print(missing_by_year.to_string())

    print("\nFirst 10 missing months:")
    for month in miss_months[:10]:
        print(f"  - {month.strftime('%B %Y')}")

    if len(miss_months) > 10:
        print(f"  ... and {len(miss_months) - 10} more")

    # Check if missing months are at the beginning or end
    if miss_months[0] == all_months[0]:
        print("\n Missing data at the beginning of the time series")
    if miss_months[-1] == all_months[-1]:
        print("\n Missing data at the end of the time series")
else:
    print("\n No missing months detected. The dataset is perfectly continuous!")

MISSING MONTHS ANALYSIS

Missing months by year:
year
2005    4

First 10 missing months:
  - June 2005
  - July 2005
  - August 2005
  - September 2005
